# Web summurizer using openAI API key

In [34]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

# If you get an error running this cell, then please head over to the troubleshooting notebook!

- check if the Api is available in .env

In [35]:

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


- quick call to frontier model, for preview

In [37]:
message = "Hello, GPT! This is my first message to you."

messages = [{"role": "user", "content": message}]
messages


[{'role': 'user', 'content': 'Hello, GPT! This is my first message to you.'}]

In [38]:
openai = OpenAI()

response = openai.chat.completions.create(
    model="gpt-5-nano",
    messages=messages
)
response.choices[0].message.content

'Hi! Nice to meet you. I’m here to help with questions, tasks, and ideas. I can explain topics, write and edit, code, plan, translate, summarize, brainstorm, and more.\n\nWhat would you like to do today? If you’re not sure, here are quick options:\n- Learn something new: ask me to explain a topic in simple terms.\n- Create or edit: emails, resumes, stories, poems, etc.\n- Solve a problem: math, coding, puzzles.\n- Plan or organize: trips, study plans, project timelines.\n- Get ideas: names, slogans, recipes, workouts.\n- Translate or summarize: articles or documents.\n\nTell me your goal or how you’d like me to respond (concise, detailed, formal, casual), and we’ll take it from there. What’s on your mind?'

In [39]:
ed = fetch_website_contents("https://edwarddonner.com")
print(ed)

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
F

In [26]:
# Let's try one out. Change the website and add print statements to follow along.

ed = Website("https://edwarddonner.com")
print(ed.title)
print(ed.text)

Home - Edward Donner
Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with 900,000 enrollments across 194 countries. The
full curriculum is here
. If you’re visiting from one of my courses – I’m super grateful!
Fo

- system_prompt

In [40]:

#define the system prompt for the LLM to summarize the website content
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

- User prompt 

In [41]:
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

- using standard form sending request to on API of the OpenAI

In [45]:
messages = [{"role": "system", "content": "you are a talkative assistant"},
 
 {"role": "user", "content": "what is 2 + 2?"}
 ]

response=openai.chat.completions.create(model="gpt-5-nano", messages=messages)

response.choices[0].message.content

'4. In base-10 arithmetic, 2 + 2 equals 4. Want a quick explanation or see how this looks in binary or other operations?'

# creating function for sending messages

In [47]:
def messages_for(website):
    return[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [48]:
messages_for(ed)

[{'role': 'system',
  'content': 'You are an assistant that analyzes the contents of a website and provides a short summary, ignoring text that might be navigation related. Respond in markdown.'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director

# now here's function for summurizing the website

In [53]:
def summarize(url):
    website=fetch_website_contents(url)
    response=openai.chat.completions.create(
        model="gpt-5-nano",
        messages=messages_for(website)
    )
    
    return response.choices[0].message.content

# now calling the summarize function

In [54]:
summarize("https://edwarddonner.com")

'- The website is a personal AI-focused portfolio and blog for Ed (Edward Donner). It combines a bio, career highlights, and a library of AI/LLM resources with a particular emphasis on building and deploying AI systems.\n\n- Key elements:\n  - Bio and background: Co-founder/CTO of Nebula.io; former founder/CEO of untapt (acquired 2021); former JPMorgan Managing Director.\n  - Education content: Udemy courses with broad reach (approximately 900,000 enrollments across 194 countries) and a link to a full curriculum.\n  - Projects: A project/section called Outsmart described as an arena where LLMs compete in diplomacy and deviousness.\n  - Resources and posts: A Resources section with posts about AI tooling and deployment, plus a blog-like Posts area.\n\n- Recent announcements (news items):\n  - February 17, 2026: AI Coder: Vibe Coder to Agentic Engineer – RESOURCES\n  - January 4, 2026: AI Builder with n8n – Create Agents and Voice Agents – RESOURCES\n  - September 15, 2025: AI Engineerin

# Now we are gonna display it in a readable form

In [55]:
def display_summary(url):
    summary=summarize(url)
    display(Markdown(summary))

# Callig function
- the display_summary(url)
- and pass the actual URL of website funtion
- url = "https://example.com"

In [57]:
display_summary("https://openai.com")

OpenAI | Research & Deployment serves as OpenAI’s hub for research, products, and deployment updates, along with company news and real-world stories. It aggregates recent research papers, product announcements (e.g., GPT-6.1 Sol, GPT-6 Astra, Introducing dots, MentalHealthBench), and various news items about OpenAI initiatives, leadership, and industry impact. It also features case studies and stories about how ChatGPT is being used in different domains, plus safety and deployment information.

Key sections you’ll find:
- Research, Products, Business, Developers, Company, Foundation
- Stories (practical applications and use cases)
- Latest research and publications
- News and announcements (recent organizational updates, launches, and leadership)

Recent news and announcements (highlights):
- Two years of OpenAI Academy (Sep 23, 2026)
- Advisory group on mathematics and artificial intelligence (Sep 21, 2026)
- Introducing Astra for Law (Sep 17, 2026)
- Paul Christiano joins OpenAI Foundation Board (Sep 9, 2026)
- The Work Now Within Reach (Sep 8, 2026)
- Supporting journalism from classrooms to newsrooms (Sep 8, 2026)

Stories and research highlights:
- Stories: Using ChatGPT to grow ATV Big Air Tour; Supporting the blind community with ChatGPT; Accelerating antibiotic discovery with ChatGPT
- Latest research: An OpenAI model proposes a solution to the Navier–Stokes problem; The view inside OpenAI; Ten advances in mathematics and theoretical computer science
- Product/industry impacts: Model ML completes finance work more efficiently with GPT-5.6 Sol; Virgin Atlantic and Zapier use ChatGPT Work

Overall, it’s a centralized portal for OpenAI’s ongoing research, product developments, deployment safety, business news, and real-world impact stories.